# Olympics Dataset Merge

This notebook merges the following datasets:

- athlete_events.csv
- noc_regions.csv
- gdp_per_capita.csv
- pop_count.csv

Steps:
1. Merge athletes with NOC regions
2. Clean country names
3. Merge GDP per capita by country and year
4. Merge population count by country and year
5. Save final dataset

In [7]:
import pandas as pd
import unicodedata

In [ ]:
# ----------------------------
# Helper functions
# ----------------------------
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )
    return df


def clean_text(series):
    return (
        series.fillna("")
        .astype(str)
        .str.strip()
        .map(lambda x: unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode("utf-8"))
        .str.lower()
        .str.replace("-", " ", regex=False)
        .str.replace(r"[.,']", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


# ----------------------------
# Load data
# ----------------------------
athletes = pd.read_csv("../data/athlete_events.csv")
regions = pd.read_csv("../data/noc_regions.csv")
gdp = pd.read_csv("../data/gdp_per_capita.csv", skiprows=4, encoding="utf-8-sig")
pop = pd.read_csv("../data/pop_count.csv")


# ----------------------------
# Clean column names
# ----------------------------
athletes = clean_columns(athletes)
regions = clean_columns(regions)
gdp = clean_columns(gdp)
pop = clean_columns(pop)

for df in [athletes, regions, gdp, pop]:
    df.drop(
        columns=[col for col in df.columns if col.startswith("unnamed")],
        inplace=True,
        errors="ignore"
    )


# ----------------------------
# Standardize base columns
# ----------------------------
athletes["noc"] = athletes["noc"].astype(str).str.strip().str.upper()
regions["noc"] = regions["noc"].astype(str).str.strip().str.upper()
athletes["year"] = pd.to_numeric(athletes["year"], errors="coerce")

regions["region_clean"] = clean_text(regions["region"])


# ----------------------------
# Merge athletes + NOC regions
# ----------------------------
regions_unique = (
    regions[["noc", "region", "region_clean"]]
    .drop_duplicates(subset=["noc"])
)

olympics = athletes.merge(
    regions_unique,
    on="noc",
    how="left"
)


# ----------------------------
# Rename columns for your EDA notebook
# ----------------------------
olympics = olympics.rename(columns={
    "region": "country",
    "sex": "gender"
})


# ----------------------------
# Prepare GDP from wide -> long
# ----------------------------
gdp_year_cols = [col for col in gdp.columns if str(col).isdigit()]

gdp_long = gdp.melt(
    id_vars=["country_name", "country_code", "indicator_name", "indicator_code"],
    value_vars=gdp_year_cols,
    var_name="year",
    value_name="gdp_per_capita"
)

gdp_long["year"] = pd.to_numeric(gdp_long["year"], errors="coerce")
gdp_long["gdp_per_capita"] = pd.to_numeric(gdp_long["gdp_per_capita"], errors="coerce")
gdp_long["country_clean"] = clean_text(gdp_long["country_name"])

gdp_long = gdp_long[
    ["country_name", "country_code", "country_clean", "year", "gdp_per_capita"]
].drop_duplicates(subset=["country_code", "year"])


# ----------------------------
# Prepare population from wide -> long
# ----------------------------
pop_year_cols = [col for col in pop.columns if str(col).isdigit()]

pop_long = pop.melt(
    id_vars=["country_name", "country_code", "indicator_name", "indicator_code"],
    value_vars=pop_year_cols,
    var_name="year",
    value_name="population"
)

pop_long["year"] = pd.to_numeric(pop_long["year"], errors="coerce")
pop_long["population"] = pd.to_numeric(pop_long["population"], errors="coerce")
pop_long["country_clean"] = clean_text(pop_long["country_name"])

pop_long = pop_long[
    ["country_name", "country_code", "country_clean", "year", "population"]
].drop_duplicates(subset=["country_code", "year"])


# ----------------------------
# Build World Bank code mapping from Olympic NOC
# ----------------------------
wb_code_map = {
    "USA": "USA",
    "GBR": "GBR",
    "FRA": "FRA",
    "GER": "DEU",
    "ITA": "ITA",
    "ESP": "ESP",
    "CAN": "CAN",
    "AUS": "AUS",
    "JPN": "JPN",
    "CHN": "CHN",
    "KOR": "KOR",
    "PRK": "PRK",
    "RUS": "RUS",
    "POL": "POL",
    "ROU": "ROU",
    "BUL": "BGR",
    "HUN": "HUN",
    "CZE": "CZE",
    "SVK": "SVK",
    "SRB": "SRB",
    "CRO": "HRV",
    "SLO": "SVN",
    "LAT": "LVA",
    "EST": "EST",
    "LTU": "LTU",
    "UKR": "UKR",
    "BLR": "BLR",
    "MDA": "MDA",
    "KAZ": "KAZ",
    "UZB": "UZB",
    "KGZ": "KGZ",
    "TJK": "TJK",
    "TKM": "TKM",
    "ARM": "ARM",
    "AZE": "AZE",
    "GEO": "GEO",
    "MGL": "MNG",
    "CUB": "CUB",
    "LBN": "LBN",
    "SMR": "SMR",
    "PUR": "PRI",
    "GUM": "GUM",
    "ISV": "VIR",
    "IVB": "VGB",
    "TTO": "TTO",
    "BAH": "BHS",
    "GAM": "GMB",
    "EGY": "EGY",
    "IRI": "IRN",
    "SYR": "SYR",
    "TUR": "TUR",
    "HKG": "HKG",
    "MAC": "MAC",
    "LAO": "LAO",
    "VIE": "VNM",
    "BRN": "BRN",
    "COD": "COD",
    "CGO": "COG",
    "CIV": "CIV",
    "ANT": "ATG",
    "SKN": "KNA",
    "TAN": "TZA",
    "RSA": "ZAF",
    "SUI": "CHE",
    "NED": "NLD",
    "DEN": "DNK",
    "NOR": "NOR",
    "SWE": "SWE",
    "AUT": "AUT",
    "BEL": "BEL",
    "FIN": "FIN",
    "GRE": "GRC",
    "POR": "PRT",
    "MEX": "MEX",
    "BRA": "BRA",
    "ARG": "ARG",
    "CHI": "CHL",
    "COL": "COL",
    "PER": "PER",
    "VEN": "VEN",
    "BOL": "BOL",
    "PAR": "PRY",
    "URU": "URY",
    "ECU": "ECU",
    "PAN": "PAN",
    "CRC": "CRI",
    "DOM": "DOM",
    "JAM": "JAM",
    "CYP": "CYP",
    "ISR": "ISR",
    "IND": "IND",
    "PAK": "PAK",
    "SRI": "LKA",
    "NEP": "NPL",
    "THA": "THA",
    "MAS": "MYS",
    "SGP": "SGP",
    "INA": "IDN",
    "PHI": "PHL",
    "MOZ": "MOZ",
    "MKD": "MKD",
}

olympics["wb_code"] = olympics["noc"].map(wb_code_map)
olympics["country_clean"] = clean_text(olympics["country"])


# ----------------------------
# Country-name fallback fixes
# ----------------------------
country_fix = {
    "usa": "united states",
    "uk": "united kingdom",
    "great britain": "united kingdom",
    "south korea": "korea rep",
    "north korea": "korea dem peoples rep",
    "russia": "russian federation",
    "iran": "iran islamic rep",
    "syria": "syrian arab republic",
    "venezuela": "venezuela rb",
    "moldova": "moldova",
    "tanzania": "tanzania",
    "laos": "lao pdr",
    "vietnam": "viet nam",
    "brunei": "brunei darussalam",
    "czech republic": "czechia",
    "slovakia": "slovak republic",
    "egypt": "egypt arab rep",
    "bahamas": "bahamas the",
    "gambia": "gambia the",
    "hong kong": "hong kong sar china",
    "macao": "macao sar china",
    "taiwan": "china",
    "trinidad": "trinidad and tobago",
    "kyrgyzstan": "kyrgyz republic",
    "ivory coast": "cote d ivoire",
    "antigua": "antigua and barbuda",
    "boliva": "bolivia",
    "republic of congo": "congo rep",
    "democratic republic of the congo": "congo dem rep",
    "macedonia": "north macedonia",
    "saint kitts": "st kitts and nevis",
}

olympics["country_clean"] = olympics["country_clean"].replace(country_fix)


# ----------------------------
# First GDP merge: by World Bank code + year
# ----------------------------
olympics = olympics.merge(
    gdp_long[["country_code", "year", "gdp_per_capita"]].rename(columns={"country_code": "wb_code"}),
    on=["wb_code", "year"],
    how="left"
)


# ----------------------------
# Second GDP merge: fallback by cleaned country name
# ----------------------------
gdp_name_lookup = gdp_long[["country_clean", "year", "gdp_per_capita"]].rename(
    columns={"gdp_per_capita": "gdp_per_capita_name"}
)

olympics = olympics.merge(
    gdp_name_lookup,
    on=["country_clean", "year"],
    how="left"
)

olympics["gdp_per_capita"] = olympics["gdp_per_capita"].fillna(olympics["gdp_per_capita_name"])
olympics = olympics.drop(columns=["gdp_per_capita_name"], errors="ignore")


# ----------------------------
# First population merge: by World Bank code + year
# ----------------------------
olympics = olympics.merge(
    pop_long[["country_code", "year", "population"]].rename(columns={"country_code": "wb_code"}),
    on=["wb_code", "year"],
    how="left"
)


# ----------------------------
# Second population merge: fallback by cleaned country name
# ----------------------------
pop_name_lookup = pop_long[["country_clean", "year", "population"]].rename(
    columns={"population": "population_name"}
)

olympics = olympics.merge(
    pop_name_lookup,
    on=["country_clean", "year"],
    how="left"
)

olympics["population"] = olympics["population"].fillna(olympics["population_name"])
olympics = olympics.drop(columns=["population_name"], errors="ignore")


# ----------------------------
# Diagnostics
# ----------------------------
print("Overall GDP match rate:", olympics["gdp_per_capita"].notna().mean())
print("Missing GDP values:", olympics["gdp_per_capita"].isna().sum())

print("Overall population match rate:", olympics["population"].notna().mean())
print("Missing population values:", olympics["population"].isna().sum())

post_1960 = olympics[olympics["year"] >= 1960]
print("Post-1960 GDP match rate:", post_1960["gdp_per_capita"].notna().mean())
print("Post-1960 population match rate:", post_1960["population"].notna().mean())

print("\nTop unmatched GDP countries after 1960:")
print(
    post_1960.loc[post_1960["gdp_per_capita"].isna(), "country"]
    .dropna()
    .value_counts()
    .head(30)
)

print("\nTop unmatched population countries after 1960:")
print(
    post_1960.loc[post_1960["population"].isna(), "country"]
    .dropna()
    .value_counts()
    .head(30)
)


# ----------------------------
# Final cleanup
# ----------------------------
olympics = olympics.drop(
    columns=["region_clean", "country_clean", "wb_code"],
    errors="ignore"
)

olympics = olympics[
    [
        "id", "name", "gender", "age", "height", "weight",
        "team", "noc", "country", "population",
        "year", "season", "city",
        "sport", "event", "medal",
        "gdp_per_capita"
    ]
]


# ----------------------------
# Save final dataset
# ----------------------------
olympics.to_csv("../data/olympics.csv", index=False)

print("\nFinal dataset shape:", olympics.shape)
display(olympics.head())

Overall GDP match rate: 0.7071142979388896
Missing GDP values: 79406
Overall population match rate: 0.7594793372578528
Missing population values: 65209
Post-1960 GDP match rate: 0.9169568378357695
Post-1960 population match rate: 0.9848616744470804

Top unmatched GDP countries after 1960:
country
Russia                         3943
Poland                         2606
Czech Republic                 2245
Romania                        1748
Serbia                         1685
Bulgaria                       1072
North Korea                     807
Hungary                         677
Cuba                            324
Virgin Islands, US              251
Mongolia                        195
Lebanon                         175
San Marino                      155
Latvia                          120
Estonia                          97
Individual Olympic Athletes      94
Guam                             88
Lithuania                        80
Vietnam                          77
Swaziland         

,id,name,gender,age,height,weight,team,noc,country,population,year,season,city,sport,event,medal,gdp_per_capita
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,China,1.164970e+09,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN,367.822652
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,China,1.350695e+09,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN,6405.057424
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,Denmark,NaN,1920,Summer,Antwerpen,Football,Football Men's Football,NaN,NaN
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,Denmark,NaN,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold,NaN
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,Netherlands,1.476009e+07,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,NaN,17770.616238
